### Notes: Building End-to-End AI Agents in LangChain

### Topic 1: What is an AI Agent? (Conceptual)

An AI Agent is an intelligent system that solves complex, multi-step problems autonomously. Unlike a standard Language Model (LLM) that gives a single response, an agent can plan, reason, and execute a sequence of actions to achieve a high-level goal given by a user.

**The Problem AI Agents Solve:**
<img src="Screenshot 2026-09-05 at 11.44.51 PM.png">
Traditional websites and apps require users to perform many manual steps. For example, planning a trip from Delhi to Goa involves:
1.  **Booking Travel:** Searching on IRCTC/MakeMyTrip for flights/trains, comparing prices, and booking.
2.  **Booking Stay:** Searching for hotels, checking reviews, prices, and availability, then booking.
3.  **Planning Activities:** Researching top attractions, creating a daily itinerary, and booking local transport or entry tickets.
This process is time-consuming and hectic.

**How an AI Agent Solves This:**
<img src="Screenshot 2026-09-05 at 11.47.13 PM.png">
Instead of the user doing all the steps, an AI Agent acts as a personal assistant. You give it a high-level goal like, "Create a budget travel itinerary from Delhi to Goa from 1st to 7th May." The agent then:
1.  **Understands the Goal:** It breaks down the user's request into specific tasks (travel, stay, activities).
<img src="Screenshot 2026-09-05 at 11.47.27 PM.png">
2.  **Uses Tools:** It uses tools (like APIs for IRCTC, hotel booking, weather) to gather information.
3.  **Plans and Executes:** It sequentially books the cheapest travel option, finds a budget-friendly hostel, rents a scooter, plans a day-wise itinerary, and finally books everything.
<img src="Screenshot 2026-09-05 at 11.49.12 PM.png">
4.  **Maintains Context:** It remembers user preferences (e.g., "budget travel," "3AC train ticket") throughout the process.
<img src="Screenshot 2026-09-05 at 11.49.47 PM.png">
<img src="Screenshot 2026-09-05 at 11.50.16 PM.png">
5.  **Adapts:** It can re-plan if new information arises (e.g., a public holiday).
<img src="Screenshot 2026-09-05 at 11.50.33 PM.png">
<img src="Screenshot 2026-09-05 at 11.50.55 PM.png">
**User Experience:** The user simply provides preferences and confirms steps, making the entire process seamless and effortless.

---

### Topic 2: Technical Definition & Core Characteristics of an AI Agent

**Technical Definition:**
An AI Agent is an intelligent system that receives a **high-level goal** from a user and **autonomously plans, decides, and executes** a sequence of actions by using external **tools**, **APIs**, and **knowledge sources**, all while maintaining context, reasoning over multiple steps, adapting to new information, and optimizing for the intended outcome.

**Core Characteristics:**
1.  **Goal-Driven:** You only need to specify *what* to do, not *how* to do it.
2.  **Planner:** It can break down a complex problem into smaller, manageable steps.
3.  **Tool-Aware:** It knows which tools (APIs, databases, search engines) it has access to and when to use them.
4.  **Context-Aware:** It maintains a memory (context) of the conversation and its past actions.
5.  **Adaptive:** It can adjust its plan if something goes wrong or new information is received.

<img src="Screenshot 2026-09-05 at 11.54.50 PM.png">

**Key Difference from a Simple LLM:**
- **LLM:** Good at reasoning and generating text. Cannot perform actions.
- **AI Agent:** = **LLM (Reasoning Engine)** + **Tools (Action Executors)**

---

### Topic 3: The ReAct Design Pattern

**What is ReAct?**
ReAct stands for **Rea**soning + **Act**ing. It's a popular design pattern for AI agents where the agent interleaves internal reasoning (thoughts) with external actions in a structured, multi-step loop.

**How ReAct Works: The Thought-Action-Observation Loop**
The agent operates in a loop, repeating three steps until it has a final answer:

1.  **Thought:** The agent reasons about what to do next based on the user's query and its past actions.
2.  **Action:** The agent decides which tool to use and what input to give it.
3.  **Observation:** The result from the tool is captured.

**Example: "Tell me the population of the capital of France."**

- **Iteration 1:**
    - **Thought:** "I need to find the capital of France first."
    - **Action:** Use the `search` tool with input "capital of France".
    - **Observation:** "Paris"
- **Iteration 2:**
    - **Thought:** "Now I know the capital is Paris. I need to find its population."
    - **Action:** Use the `search` tool with input "population of Paris".
    - **Observation:** "2.1 million"
- **Iteration 3:**
    - **Thought:** "I now know the final answer."
    - **Final Answer:** "Paris is the capital of France and it has a population of 2.1 million."

This loop is managed by the **Agent Executor**, which orchestrates the communication between the agent (LLM) and the tools.

---

### Topic 4: Building a Simple AI Agent in LangChain (Code)

This code creates a basic agent that can search the internet using DuckDuckGo.

```python
# Run these in your terminal or command prompt
# pip install langchain-openai langchain-community duckduckgo-search
```

In [ ]:
# ----------Step 1: Import Libraries and Set Up API Keys-------
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os


load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")


#--------------- Step 2: Create a Tool-----------------
# We create a tool for internet search.
# Create the search tool
search_tool = DuckDuckGoSearchRun()

# Test the tool (optional)
# result = search_tool.invoke("Top news in India today")
# print(result)

#--------------  Step 3: Create an LLM -------------------
#We create the LLM that will act as the agent's reasoning engine.
# Create a Chat Model instance
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=api_key)
# Test the LLM (optional)
# response = model.invoke("Hi")
# print(response.content)


#-------------- Step 4: Create the Agent-------------------
# We use a pre-built prompt from LangChain Hub for the ReAct pattern. An agent is created by combining the LLM, the tools, and the prompt.

# Pull the predefined prompt that follows ReAct pattern prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")

#agent is the "brain" or the planner.It does not run the tool itself.
agent = create_react_agent(
    llm=model,
    tools=[search_tool],  # Provide the list of tools
    prompt=prompt # recommended to use prebuilt ReAct prompts to instruct llm 
)


#-------------- Step 5: Create the Agent Executor-------------------
# The Agent Executor is responsible for running the Thought-Action-Observation loop and executing the tools.
# Create the agent executor
# This is the component that will actually run the agent's plan.
agent_executor = AgentExecutor(
    agent=agent,## needs agent to instruct agent_executor
    tools=[search_tool],  # Tools are needed again for execution
    verbose=True  # Setting verbose=True will print the agent's thoughts and actions
)

#-------------- Step 6: Run the Agent-------------------
# We can now ask the agent a question.
# Invoke the agent with a query
response = agent_executor.invoke({
    "input": "What are the three ways to reach Goa from Delhi?"
})

# Print the final answer
print(response['output'])



> Entering new AgentExecutor chain...
Thought: loking for ways to travel from Delhi to Goa
Action: duckduckgo_search
Action Input: three ways to reach Goa from Delhi transport modes flight train roadMay 4, 2026 - Flying is the fastest and easiest approach when deciding how to reach Goa. Nearly every major Indian city including Delhi, Mumbai, Bangalore, Hyderabad, Chennai, and Kochi offers flights to Goa. December 24, 2025 - Madgaon (Margao) and Vasco da Gama are Goa’s two main railway stations, and they are both connected to important Indian cities. Particularly along the Konkan Railway, one can enjoy breathtaking vistas of the Western Ghats’ luxuriant vegetation, rivers and tunnels. Many express and passenger trains run daily for passengers arriving from Delhi, Bangalore, Mumbai, Pune and other nearby cities. For a modern and fast rail experience, the Mumbai-Madgaon Vande Bharat Express is now the top choice for travellers. 2 weeks ago - Learn how to reach Goa by flight, train, bus 

In [9]:
response['output']

"The three main ways to reach Goa from Delhi are:\n1. **By Air (Flight):** This is the fastest and most convenient method, with regular direct and connecting flights operating from Delhi to Goa's airports (Dabolim Airport and Mopa Airport).\n2. **By Train:** A budget-friendly option where several express and passenger trains run from Delhi to Goa's main railway stations (such as Madgaon and Vasco da Gama), offering scenic views along the Konkan Railway route.\n3. **By Road:** You can also travel by road (driving a personal vehicle, booking a taxi, or taking a long-distance bus), though it is the longest route compared to flights and trains."

**Expected Output (Conceptual):**
The `verbose=True` will print the agent's thought process. The final output will be something like:
```
The three common ways to reach Goa from Delhi are by air (flight), by train, and by road (bus or car).
```

---

### Topic 5: Understanding the Code (Agent vs. Agent Executor)

A common source of confusion is the difference between the `agent` and the `agent_executor`.

- **Agent (created by `create_react_agent`):**
    - This is the "brain" or the planner.
    - It takes the user query and the current context (thought trace).
    - It decides the next step: either to output a **final answer** or to output an **action** (which tool to use and with what input).
    - It does not run the tool itself.

- **Agent Executor (created by `AgentExecutor`):**
    - This is the "hands" or the worker.
    - It manages the entire Thought-Action-Observation loop.
    - It receives the user's query, sends it and the current trace to the agent.
    - If the agent returns an action, the executor runs the specified tool, gets the observation, updates the trace, and starts the loop again.
    - If the agent returns a final answer, the executor stops the loop and returns the answer to the user.

---

### Topic 6: Improving the Agent - Adding a 2nd tool , a Custom Tool

We can give our agent more capabilities by creating custom tools. Here, we'll add a tool that fetches current weather data for a city using the Weatherstack API.

#### Step 1: Create the Custom Tool
We use the `@tool` decorator from LangChain to turn a function into a tool.

https://api.weatherstack.com/current
    ? access_key = YOUR_ACCESS_KEY
    & query = New York

In [11]:
# --------------  Step 1: Create the Custom Tool---------------------------
import requests
from langchain.tools import tool
import os
from dotenv import load_dotenv
load_dotenv()
weather_api_key=os.getenv("weather_api_key")
api_key = os.getenv("GOOGLE_API_KEY")
#  Weatherstack API key (sign up for a free account)
@tool
def get_weather_data(city:str)->str:
    """Fetches the current weather data for a given city."""
    url=f"https://api.weatherstack.com/current?access_key={weather_api_key}&query={city}"

    try:
        response=response.get(url)
        data= response.json()
        return data
    except Exception as e:
        return f"An error occurred while fetching weather data: {e}"

# -------------- Step 2: Use the New Tool with the Agent---------------------------
# We now create a new list of tools that includes both the search tool and our custom weather tool.
from langchain_google_genai import ChatGoogleGenerativeAI

# Create a Chat Model instance
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=api_key)

# Create the tools list
tools = [search_tool, get_weather_data]

# Create the agent with the new tools
agent = create_react_agent(
    llm=model,
    tools=tools,
    prompt=hub.pull("hwchase17/react")
)

# Create the agent executor with the new tools
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# Ask a question that requires multiple steps and uses both tools

response = agent_executor.invoke({
    "input": "Find the capital of Madhya Pradesh, then find its current weather condition."
})

print(response)



> Entering new AgentExecutor chain...
Thought: I need to find the capital of Madhya Pradesh first, and then check its current weather.
Action: duckduckgo_search
Action Input: capital of Madhya PradeshIndore (/ ɪnˈdɔːr / ⓘ; ISO: Indaura, Hindi: [ɪn̪d̪ɔːr]) is the largest city of the Indian state of Madhya Pradesh where it serves as the capital and the administrative headquarters of the eponymous district and division. Modern-day Indore was established on the banks of the Kanh and Saraswati rivers and traces its roots to its 16th-century founding as a trading hub between the ... This page was last edited on 31 August 2026, at 05:45 (UTC). Madhya Pradesh, state of India that is situated in the heart of the country. It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive plateaus, and river valleys. The capital is Bhopal, in the west-central part of the state. Bhopal is the capital city of Madhya Pradesh state, central India, Situated 

In [12]:
response["output"]

'The capital of Madhya Pradesh is Bhopal. The current weather in Bhopal is approximately 22.8°C with rain showers or fog.'

**Expected Output (Conceptual):**
With `verbose=True`, you will see the agent's entire thought process:
1.  **Thought:** "First, I need to find the capital of Madhya Pradesh."
    - **Action:** Use `DuckDuckGoSearchRun` with input "capital of Madhya Pradesh".
    - **Observation:** "The capital of Madhya Pradesh is Bhopal."
2.  **Thought:** "Now I know the capital is Bhopal. I can use the `get_weather_data` tool to check its current weather condition."
    - **Action:** Use `get_weather_data` with input "Bhopal".
    - **Observation:** "The weather in Bhopal is partly cloudy with a temperature of 40°C, humidity of 30%, and wind speed of 10 km/h."
3.  **Thought:** "I now know the final answer."
    - **Final Answer:** "The capital of Madhya Pradesh is Bhopal, and the current weather condition is partly cloudy with a temperature of 40°C."

---

### Topic 7: Conclusion and Future Directions

- **Summary:** We learned that AI Agents are autonomous systems that use an LLM for reasoning and tools for action, following patterns like ReAct (Thought-Action-Observation) to solve complex, multi-step problems.
- **Current State of LangChain for Agents:** While LangChain provides the basic building blocks (`create_react_agent`, `AgentExecutor`), it is not the recommended library for building scalable, production-grade AI agents.
- **What's Next:** For building truly powerful and scalable agents, you should learn **LangGraph**. LangGraph is a library from the LangChain team designed specifically for building robust, stateful, and controllable agent workflows.